### Xử lý dữ liệu

Sau khi xem qua dữ liệu một lượt, thấy rằng có nhiều vật cản để xác định và tìm ra được hắc tố có thể kể đến như: tóc, lông, vết đo, bị tối,.... 

In [11]:
!pip install albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 1.0 MB/s eta 0:00:0000:010m00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 1.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.14.6
    Uninstalling pydantic_core-2.14.6:
      Successfully uninstalled pydantic_core-2.14.6
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.5.3
    Uninstalling pydantic-2.5.3:
      Successfully uninstalled pydantic-2.5.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtext 0.16.0 requires torch==2.1.0, but you have torch 2.7.0 which is incompatible.
torchdata 0.7.0 requ

In [2]:
!pip install opencv-python --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 2.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.8.0.74
    Uninstalling opencv-python-4.8.0.74:
      Successfully uninstalled opencv-python-4.8.0.74

[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg19
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
import cv2
import numpy as np
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm

ModuleNotFoundError: No module named 'albumentations'

In [ ]:
import cv2

#IMAGE ACQUISITION

#Input image
path='ISIC_0031023.jpg'
#Read image
image=cv2.imread(path,cv2.IMREAD_COLOR)
#Image cropping
img=image[30:410,30:560]
    
#DULL RAZOR (REMOVE HAIR)

#Gray scale
grayScale = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY )
#Black hat filter
kernel = cv2.getStructuringElement(1,(9,9)) 
blackhat = cv2.morphologyEx(grayScale, cv2.MORPH_BLACKHAT, kernel)
#Gaussian filter
bhg= cv2.GaussianBlur(blackhat,(3,3),cv2.BORDER_DEFAULT)
#Binary thresholding (MASK)
ret,mask = cv2.threshold(bhg,10,255,cv2.THRESH_BINARY)
#Replace pixels of the mask
dst = cv2.inpaint(img,mask,6,cv2.INPAINT_TELEA)   

#Display images
cv2.imshow("Original image",image)
cv2.imshow("Cropped image",img)
cv2.imshow("Gray Scale image",grayScale)
cv2.imshow("Blackhat",blackhat)
cv2.imshow("Binary mask",mask)
cv2.imshow("Clean image",dst)

cv2.waitKey()
cv2.destroyAllWindows()

2025-06-11 06:31:32.957 python[48279:19065341] +[IMKClient subclass]: chose IMKClient_Modern
2025-06-11 06:31:32.957 python[48279:19065341] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [ ]:
class SkinLesionDataset(torch.utils.data.Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images
        self.masks = masks
        self.transform = transform
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, index):
        image = cv2.imread(self.images[index])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(self.masks[index], cv2.IMREAD_GRAYSCALE)
        mask = mask / 255.0
        
        image = self.remove_artifacts(image)
        
        if self.transform:
            agumented = self.transform(image=image, mask=mask)
            image = agumented["image"]
            mask = agumented["mask"]
    
        image = torch.from_numpy(image.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()
        
        return image, mask
        
    def remove_artifacts(self, image):
        #Gray scale
        grayScale = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY )
        #Black hat filter
        kernel = cv2.getStructuringElement(1, (9,9)) 
        blackhat = cv2.morphologyEx(grayScale, cv2.MORPH_BLACKHAT, kernel)
        #Gaussian filter
        bhg = cv2.GaussianBlur(blackhat, (3,3), cv2.BORDER_DEFAULT)
        #Binary thresholding (MASK)
        _, mask = cv2.threshold(bhg, 10, 255, cv2.THRESH_BINARY)
        #Replace pixels of the mask
        reuslt = cv2.inpaint(image, mask, 6, cv2.INPAINT_TELEA)   
        return reuslt

Tóc hay lông trong ảnh sẽ được xử lý bằng xử lý ảnh thuần túy:
- A

Các vết khác như mực hay vết thước kẻ do tính đặc thù và ít xuất hiện hơn nên sẽ được bỏ qua

### Model

Với hiểu biết vốn ban đầu của em về phân đoạn ảnh y tế, mô hình có thể thực hiện tác vụ này là U-net 

In [9]:
class Conv2D(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, padding=1, dilation=1, bias=False, act=True):
        super().__init__()
        self.act = act

        self.conv = nn.Sequential(
            nn.Conv2d(
                in_c, out_c,
                kernel_size=kernel_size,
                padding=padding,
                dilation=dilation,
                bias=bias
            ),
            nn.BatchNorm2d(out_c)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        if self.act == True:
            x = self.relu(x)
        return x

class squeeze_excitation_block(nn.Module):
    def __init__(self, in_channels, ratio=8):
        super().__init__()

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, max(in_channels//ratio, 8)),  # Ensure minimum channels
            nn.ReLU(inplace=True),
            nn.Linear(max(in_channels//ratio, 8), in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        batch_size, channel_size, _, _ = x.size()
        y = self.avgpool(x).view(batch_size, channel_size)
        y = self.fc(y).view(batch_size, channel_size, 1, 1)
        return x * y.expand_as(x)

class ASPP(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.avgpool = nn.Sequential(
            nn.AdaptiveAvgPool2d((2, 2)),
            Conv2D(in_c, out_c, kernel_size=1, padding=0)
        )

        self.c1 = Conv2D(in_c, out_c, kernel_size=1, padding=0, dilation=1)
        self.c2 = Conv2D(in_c, out_c, kernel_size=3, padding=6, dilation=6)
        self.c3 = Conv2D(in_c, out_c, kernel_size=3, padding=12, dilation=12)
        self.c4 = Conv2D(in_c, out_c, kernel_size=3, padding=18, dilation=18)

        self.c5 = Conv2D(out_c*5, out_c, kernel_size=1, padding=0, dilation=1)

    def forward(self, x):
        x0 = self.avgpool(x)
        x0 = F.interpolate(x0, size=x.size()[2:], mode="bilinear", align_corners=True)

        x1 = self.c1(x)
        x2 = self.c2(x)
        x3 = self.c3(x)
        x4 = self.c4(x)

        xc = torch.cat([x0, x1, x2, x3, x4], axis=1)
        y = self.c5(xc)
        return y

class conv_block(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()

        self.c1 = Conv2D(in_c, out_c)
        self.c2 = Conv2D(out_c, out_c)
        self.a1 = squeeze_excitation_block(out_c)

    def forward(self, x):
        x = self.c1(x)
        x = self.c2(x)
        x = self.a1(x)
        return x

class encoder1(nn.Module):
    def __init__(self):
        super().__init__()

        # Load VGG19 features
        vgg = vgg19(pretrained=True)
        
        self.x1 = vgg.features[:4]   # 64 channels
        self.x2 = vgg.features[4:9]  # 128 channels  
        self.x3 = vgg.features[9:18] # 256 channels
        self.x4 = vgg.features[18:27] # 512 channels
        self.x5 = vgg.features[27:36] # 512 channels

    def forward(self, x):
        x1 = self.x1(x)
        x2 = self.x2(x1)
        x3 = self.x3(x2)
        x4 = self.x4(x3)
        x5 = self.x5(x4)
        return x5, [x4, x3, x2, x1]

class decoder1(nn.Module):
    def __init__(self):
        super().__init__()

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.c1 = conv_block(512+512, 256)  # 512 from x5 + 512 from skip x4
        self.c2 = conv_block(256+256, 128)  # 256 from c1 + 256 from skip x3
        self.c3 = conv_block(128+128, 64)   # 128 from c2 + 128 from skip x2
        self.c4 = conv_block(64+64, 32)     # 64 from c3 + 64 from skip x1

    def forward(self, x, skip):
        s1, s2, s3, s4 = skip  # [x4, x3, x2, x1]

        x = self.up(x)
        x = torch.cat([x, s1], axis=1)
        x = self.c1(x)

        x = self.up(x)
        x = torch.cat([x, s2], axis=1)
        x = self.c2(x)

        x = self.up(x)
        x = torch.cat([x, s3], axis=1)
        x = self.c3(x)

        x = self.up(x)
        x = torch.cat([x, s4], axis=1)
        x = self.c4(x)

        return x

class encoder2(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d((2, 2))

        self.c1 = conv_block(3, 32)
        self.c2 = conv_block(32, 64)
        self.c3 = conv_block(64, 128)
        self.c4 = conv_block(128, 256)

    def forward(self, x):
        x1 = self.c1(x)
        p1 = self.pool(x1)

        x2 = self.c2(p1)
        p2 = self.pool(x2)

        x3 = self.c3(p2)
        p3 = self.pool(x3)

        x4 = self.c4(p3)
        p4 = self.pool(x4)

        return p4, [x4, x3, x2, x1]

class decoder2(nn.Module):
    def __init__(self):
        super().__init__()

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        # Skip connections from both encoders
        self.c1 = conv_block(256+512+256, 256)  # 256 from encoder2 + 512 from encoder1 + 256 from encoder2 skip
        self.c2 = conv_block(256+256+128, 128)  # 256 from c1 + 256 from encoder1 + 128 from encoder2 skip
        self.c3 = conv_block(128+128+64, 64)    # 128 from c2 + 128 from encoder1 + 64 from encoder2 skip
        self.c4 = conv_block(64+64+32, 32)      # 64 from c3 + 64 from encoder1 + 32 from encoder2 skip

    def forward(self, x, skip1, skip2):
        s1_1, s1_2, s1_3, s1_4 = skip1  # From encoder1: [x4, x3, x2, x1]
        s2_1, s2_2, s2_3, s2_4 = skip2  # From encoder2: [x4, x3, x2, x1]

        x = self.up(x)
        x = torch.cat([x, s1_1, s2_1], axis=1)
        x = self.c1(x)

        x = self.up(x)
        x = torch.cat([x, s1_2, s2_2], axis=1)
        x = self.c2(x)

        x = self.up(x)
        x = torch.cat([x, s1_3, s2_3], axis=1)
        x = self.c3(x)

        x = self.up(x)
        x = torch.cat([x, s1_4, s2_4], axis=1)
        x = self.c4(x)

        return x

class DoubleUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.e1 = encoder1()
        self.a1 = ASPP(512, 64)
        self.d1 = decoder1()
        self.y1 = nn.Conv2d(32, 1, kernel_size=1, padding=0)
        self.sigmoid = nn.Sigmoid()

        self.e2 = encoder2()
        self.a2 = ASPP(256, 64)
        self.d2 = decoder2()
        self.y2 = nn.Conv2d(32, 1, kernel_size=1, padding=0)

    def forward(self, x):
        # First U-Net
        x_enc1, skip1 = self.e1(x)
        x_aspp1 = self.a1(x_enc1)
        x_dec1 = self.d1(x_aspp1, skip1)
        y1 = self.y1(x_dec1)

        # Attention mechanism: multiply input with sigmoid of first output
        input_x = x * self.sigmoid(y1)
        
        # Second U-Net
        x_enc2, skip2 = self.e2(input_x)
        x_aspp2 = self.a2(x_enc2)
        x_dec2 = self.d2(x_aspp2, skip1, skip2)
        y2 = self.y2(x_dec2)

        return y1, y2

In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5, beta=0.5, gamma=2):
        super().__init__()
        self.alpha = alpha  # Weight for y1 loss
        self.beta = beta    # Weight for y2 loss
        self.gamma = gamma  # Focal loss gamma
        
    def dice_loss(self, pred, target):
        smooth = 1e-6
        pred_flat = pred.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        return 1 - (2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)
    
    def focal_loss(self, pred, target):
        bce_loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()
    
    def forward(self, y1, y2, target):
        # Apply sigmoid to predictions
        y1_sig = torch.sigmoid(y1)
        y2_sig = torch.sigmoid(y2)
        
        # Calculate losses for both outputs
        dice1 = self.dice_loss(y1_sig, target)
        dice2 = self.dice_loss(y2_sig, target)
        
        focal1 = self.focal_loss(y1, target)
        focal2 = self.focal_loss(y2, target)
        
        # Combined loss
        loss1 = dice1 + focal1
        loss2 = dice2 + focal2
        
        total_loss = self.alpha * loss1 + self.beta * loss2
        
        return total_loss, loss1, loss2


In [ ]:
def get_transforms():
    train_transform = A.Compose([
        A.RandomRotate90(p=0.5),
        A.Flip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
        A.CLAHE(clip_limit=2.0, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.Resize(384, 384),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = A.Compose([
        A.Resize(384, 384),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

In [ ]:
def calculate_dice_score(pred, target, threshold=0.5):
    pred_binary = (pred > threshold).float()
    intersection = (pred_binary * target).sum()
    union = pred_binary.sum() + target.sum()
    dice = (2.0 * intersection + 1e-6) / (union + 1e-6)
    return dice.item()

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    total_dice = 0
    
    progress_bar = tqdm(train_loader, desc="Training")
    
    for images, masks in progress_bar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        y1, y2 = model(images)
        loss, loss1, loss2 = criterion(y1, y2, masks)
        
        loss.backward()
        optimizer.step()
        
        # Calculate dice score using y2 (final output)
        with torch.no_grad():
            dice = calculate_dice_score(torch.sigmoid(y2), masks)
        
        total_loss += loss.item()
        total_dice += dice
        
        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Dice': f'{dice:.4f}'
        })
    
    return total_loss / len(train_loader), total_dice / len(train_loader)

In [ ]:
def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_dice = 0
    
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc="Validation"):
            images = images.to(device)
            masks = masks.to(device)
            
            y1, y2 = model(images)
            loss, loss1, loss2 = criterion(y1, y2, masks)
            
            dice = calculate_dice_score(torch.sigmoid(y2), masks)
            
            total_loss += loss.item()
            total_dice += dice
    
    return total_loss / len(val_loader), total_dice / len(val_loader)

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=100, lr=1e-4):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on: {device}")
    
    model.to(device)
    
    criterion = CombinedLoss(alpha=0.3, beta=0.7)  # More weight on final output
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-6)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )
    
    best_dice = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': []}
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 50)
        
        # Training
        train_loss, train_dice = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validation
        val_loss, val_dice = validate_epoch(model, val_loader, criterion, device)
        
        # Update scheduler
        scheduler.step()
        
        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        
        print(f"Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Save best model
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_dice': best_dice,
            }, 'best_doubleunet_model.pth')
            print(f"New best model saved! Dice: {best_dice:.4f}")
    
    return history